# 🌊 FloodSentinel - Sri Lanka Flood Early Warning & Risk Prediction System
## Machine Learning Pipeline (Module Assignment)

**Module:** Machine Learning Group Assignment  
**Team Member (Lead for Steps 1 & 2):** Aloka Fernando  
**Project Team:** FloodSentinel (Group of 4)  
**Domain:** Hydro-Meteorological Disaster Risk Prediction for Sri Lanka  
**Primary Dataset:** `sri_lanka_flood_risk_dataset_25000.csv` (25,000 Records, 32 Features)  
**Target Variable:** `flood_occurrence_current_event` (Binary: Yes / No)  

---

## 📍 Step 1: Problem Definition
**Author / Contributor:** Aloka Fernando  


### 1.1 Real-World Context & Disaster Landscape in Sri Lanka
Sri Lanka is an island nation heavily influenced by two major monsoon seasons:
1. **South-West Monsoon (May to September):** Brings intense precipitation to the Western, Southern, and Sabaragamuwa provinces, frequently inundating the Kelani, Kalu, Gin, and Nilwala river basins.
2. **North-East Monsoon (November to February):** Brings heavy convective rain to the Northern, Eastern, and North-Central dry zones.

Rapid urbanization, loss of natural wetlands, and inadequate drainage infrastructure have escalated flood vulnerability in dense metropolitan areas like Colombo and Gampaha, as well as downstream rural settlements. Floods inflict severe infrastructure damage, agricultural loss, and human displacement annually.

### 1.2 Objective & Machine Learning Formulation
The objective of **FloodSentinel** is to develop a robust, high-recall machine learning classification system capable of predicting active flood occurrence for any given location in Sri Lanka based on:
- **Topographic parameters** (Elevation, distance to nearest major river)
- **Meteorological indices** (7-day cumulative rainfall, monthly rainfall, natural drainage capacity)
- **Satellite Earth Observation indices** (NDVI for vegetation density, NDWI for surface water saturation)
- **Regional infrastructure & demographic metrics** (Built-up percentage, population density, road quality)

### 1.3 Mathematical Formulation & Cost-Sensitive Alert Criteria
We formulate this as a **supervised binary classification problem**:
$$\hat{y} = f(X) \in \{0, 1\}$$
where:
- $y = 0$: No flood occurrence (Normal conditions)
- $y = 1$: Active flood event (Emergency alert triggered)

#### The Asymmetric Disaster Cost Matrix:
In disaster early warning, classification errors have profoundly asymmetric consequences:
- **False Positive ($FP$):** Warning issued, but no flood occurs. Cost: Temporary evacuation inconvenience, economic friction ($C_{FP} = 1$).
- **False Negative ($FN$):** Flood occurs, but no warning is issued. Cost: Loss of human lives, destruction of property, failure of disaster response ($C_{FN} \approx 10 \times C_{FP}$).

Therefore, standard raw classification accuracy is an inappropriate metric. The primary optimization criteria are:
1. **Precision-Recall AUC (PR-AUC):** Optimal evaluation under heavy class imbalance (~9.9% positive class).
2. **Recall / Sensitivity ($> 90\%$):** Ensuring almost all genuine flood events trigger an alert.
3. **Cost-Sensitive Risk Score:** Calibrating predicted probabilities into four actionable operational tiers: `Safe`, `Advisory`, `Warning`, and `Critical Emergency`.

## 📦 Step 2: Data Collection & Ingestion
**Author / Contributor:** Aloka Fernando  


### 2.1 Dataset Provenance & Metadata
- **File:** `../sri_lanka_flood_risk_dataset_25000.csv` (or root `sri_lanka_flood_risk_dataset_25000.csv`)
- **Records:** 25,000 spatial observations
- **Attributes:** 32 features encompassing geographic coordinates, elevation, rainfall measurements, satellite indices, and infrastructure attributes across all 25 Sri Lankan administrative districts.

Let us import the necessary analytical libraries and load the raw dataset.

In [ ]:
# Import core scientific and analytical libraries
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Configure visualization styles
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

print('Libraries successfully loaded!')

In [ ]:
# Locate and load the Sri Lanka Flood Risk dataset
data_paths = [
    'sri_lanka_flood_risk_dataset_25000.csv',
    '../sri_lanka_flood_risk_dataset_25000.csv',
    'data/raw/sri_lanka_flood_risk_dataset_25000.csv'
]

dataset_path = None
for path in data_paths:
    if os.path.exists(path):
        dataset_path = path
        break

if dataset_path is None:
    raise FileNotFoundError('Could not locate sri_lanka_flood_risk_dataset_25000.csv!')

df = pd.read_csv(dataset_path)
print(f'Successfully loaded dataset from: {dataset_path}')
print(f'Total Records (Rows):    {df.shape[0]:,}')
print(f'Total Attributes (Cols): {df.shape[1]}')

In [ ]:
# Inspect the first 5 records of the dataset
df.head()

### 2.2 Feature Dictionary & Attribute Schema
Below is the complete systematic classification of all 32 columns present in the raw dataset:

In [ ]:
# Construct a structured schema audit dataframe
schema_audit = pd.DataFrame({
    'Column Name': df.columns,
    'Data Type': df.dtypes.values,
    'Non-Null Count': df.notnull().sum().values,
    'Null Count': df.isnull().sum().values,
    'Null Percentage (%)': ((df.isnull().sum() / len(df)) * 100).round(2).values,
    'Sample Value': [df[col].iloc[0] for col in df.columns]
})

schema_audit

### 2.3 Data Integrity & Quality Audit
Let us verify missing values, duplicate observations, and check the target class balance.

In [ ]:
# Check for duplicate rows
duplicates = df.duplicated().sum()
print(f'Total Duplicate Rows: {duplicates}')

# Verify missing value columns
missing_cols = schema_audit[schema_audit['Null Count'] > 0]
print(f'Columns with Missing Values: {len(missing_cols)}')
missing_cols[['Column Name', 'Null Count', 'Null Percentage (%)', 'Sample Value']]

In [ ]:
# Target Variable Analysis: flood_occurrence_current_event
target_counts = df['flood_occurrence_current_event'].value_counts(dropna=False)
target_pct = df['flood_occurrence_current_event'].value_counts(normalize=True, dropna=False) * 100

target_summary = pd.DataFrame({
    'Occurrences': target_counts,
    'Percentage (%)': target_pct.round(2)
})

print('=== TARGET VARIABLE DISTRIBUTION ===')
print(target_summary)

# Visual inspection of the class imbalance
plt.figure(figsize=(7, 4))
colors = ['#2b5c8f', '#d9534f']
ax = sns.barplot(x=target_counts.index, y=target_counts.values, palette=colors)
plt.title('Target Distribution: flood_occurrence_current_event', fontsize=13, fontweight='bold')
plt.xlabel('Flood Occurrence (Active Event)')
plt.ylabel('Number of Records')
for p in ax.patches:
    height = p.get_height()
    ax.annotate(f'{int(height):,} ({height/len(df)*100:.1f}%)',
                (p.get_x() + p.get_width() / 2., height / 2),
                ha='center', va='center', color='white', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Summary statistics for key physical, hydrologic, and satellite indicators
key_features = [
    'elevation_m', 'distance_to_river_m', 'rainfall_7d_mm', 
    'monthly_rainfall_mm', 'drainage_index', 'ndvi', 'ndwi', 
    'population_density_per_km2', 'flood_risk_score'
]
df[key_features].describe().T[['mean', 'std', 'min', '25%', '50%', '75%', 'max']].round(2)

### 2.4 Key Observations from Step 1 & Step 2
1. **Imbalance Rate (~9.89% Positive Class):** Only 2,472 out of 25,000 locations experienced active flooding during current monsoon conditions. This confirms that all subsequent modeling and cross-validation must use **stratified sampling** and **cost-sensitive learning**.
2. **Data Quality & Missing Values:**
   - `electricity` has 789 missing entries (3.16%), which will be imputed during the data cleaning stage.
   - `reason_not_good_to_live` has 18,172 missing values (72.69%), which represents post-event habitability commentary and must be dropped to prevent data leakage.
3. **Zero Duplicates:** The dataset contains 0 duplicate rows, confirming clean record indexing.
4. **Next Stages:** We are now prepared to proceed to **Step 3 (Data Understanding & Geospatial EDA)** and **Step 4 (Data Cleaning & Preprocessing)**.